# IST Landscape Pre-Processing

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
from glob import glob
import numpy as np
import pandas as pd
import scanpy as sc
import tifffile
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import geopandas as gpd
from ipywidgets import Widget

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [3]:
dataset_name = 'E12_71'
inst_slice = 'T8'

path_data = 'data/IST_data/Substrate_' + dataset_name + '/'
sample = 'Substrate_E12_71'
data_d = 'data/IST_data'
path_landscape_files=f'data/landscape_files/IST_{sample}_2025-08-04'

tile_size=200
image_tile_layer='h&e'

image_scale = 1.0
suffix = '.webp[Q=100]'

# Main Function

In [4]:
# dega.pre.main(
#     sample=sample,
#     data_root_dir=data_d,
#     tile_size=tile_size,
#     image_tile_layer=image_tile_layer,
#     path_landscape_files=path_landscape_files,
#     use_int_index=True,
# )

## Created jittered transcripts

In [5]:
# data_dir = f"{data_d}/{sample}"
# dega.pre.find_spot_positions(str(data_dir), path_landscape_files)
# df_spot_positions = pd.read_parquet(f"{path_landscape_files}/spot_positions.parquet")
# df_spot_positions.shape

In [6]:


# from celldega.pre.run_pre_processing import _setup_preprocessing_paths
# technology = "IST"
# spot_file = Path(path_landscape_files) / "spot_positions.parquet"
# paths = _setup_preprocessing_paths(technology, path_landscape_files, data_dir, sample=sample)

# dega.pre.make_pseudo_transcript_tiles(
#     paths,
#     str(spot_file),
#     str(paths["transcript_tiles"]),
#     tile_size=tile_size,
# )

# Viz

In [7]:
# meta_gene = pd.read_parquet('data/landscape_files/IST_tmp/meta_gene.parquet')
# meta_gene = meta_gene.loc[['Hbb-bs', 'Hbb-bt', 'Hbb-y', 'Hbegf', 'Tmsb10']]
# meta_gene.to_parquet('data/landscape_files/IST_tmp/meta_gene.parquet')

In [11]:
server_address = dega.viz.get_local_server()

In [12]:
Widget.close_all()

landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{server_address}/{path_landscape_files}",
    height=400
)

landscape_ist

Landscape(base_url='http://localhost:65279/data/landscape_files/IST_Substrate_E12_71_2025-08-04', cell_attr=['…

In [9]:
%%time
# pd.read_parquet('data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs_2025-05-27/cbg/ADAM28.parquet')

CPU times: user 1 μs, sys: 0 ns, total: 1 μs
Wall time: 2.86 μs


In [23]:
%%time
pd.read_parquet('data/landscape_files/Visium_HD_Human_Lung_Cancer/tbg/A2ML1.parquet')

CPU times: user 2.23 ms, sys: 1.58 ms, total: 3.81 ms
Wall time: 2.99 ms


,A2ML1
s_008um_00665_00121-1,1
s_008um_00656_00215-1,1


In [25]:
%%time
pd.read_parquet('data/landscape_files/IST_Substrate_E12_71_2025-08-01-wh/cbg/Rps27.parquet')

CPU times: user 2.14 ms, sys: 1.39 ms, total: 3.53 ms
Wall time: 4.59 ms


,Rps27
10000,9
10001,22
10002,1
10003,11
10004,14
...,...
9998,11
9999,11
999,7
99,12


In [ ]:

# tmp.set_index(0, inplace=True)
tmp.index.name = None
tmp.columns = ['x', 'y']
tmp = tmp.astype(float)
# tmp['x'] = (tmp['x'] - gc.loc[inst_slice + '_' + dataset, 'X_shift']) * high_res_scale
# tmp['y'] = (tmp['y'] - gc.loc[inst_slice + '_' + dataset, 'Y_shift']) * high_res_scale

# tmp["geometry"] = tmp.apply(
#         # swapped for some reason
#         lambda row: [row["y"], row["x"]] , axis=1
#     )

# tmp['name'] = pd.Series(tmp.index.tolist(), index=tmp.index.tolist())

# tmp[['name', 'geometry']].to_parquet('data/michal_landscape_files/E14_' + inst_slice + '/cell_metadata.parquet')

# Individual Steps

## Image

In [ ]:
# Path to your OME-TIFF file
file_path = path_data + 'registered_images/' + inst_slice + '_' + dataset_name + '.ome.tiff'

# Open the OME-TIFF file and read the image data
with tifffile.TiffFile(file_path) as tif:
    series = tif.series[0]
    image_data = series.asarray()

In [ ]:
# image_data_scaled = image_data[:,:0] * 2
# Save the image data to a regular TIFF file without compression
tifffile.imwrite(path_landscape_files + 'output_regular.tif', image_data, compression=None)
# image_ds = dega.pre.reduce_image_size(path_landscape_files + 'output_regular.tif', image_scale, path_landscape_files)
image_png = dega.pre._convert_to_png(path_landscape_files + 'output_regular.tif')
dega.pre.make_deepzoom_pyramid(image_png, path_landscape_files + 'pyramid_images/', 'h&e', suffix=suffix)

# Spots

In [ ]:
tsv_file = path_data + 'Substrate_E14_62_map_file.tsv'

In [ ]:
# Define parameters
tsv_file = path_data + 'Substrate_E12_71_map_file.tsv'
chunk_size = 10_000_000
parquet_prefix = path_landscape_files + 'map_parquet_files/output_chunk'

for i, chunk in enumerate(pd.read_csv(tsv_file, sep="\t", chunksize=chunk_size, header=None, index_col=0)):
    output_file = f"{parquet_prefix}_{i}.parquet"
    chunk.index.name = None
    chunk.to_parquet(output_file, engine="pyarrow")

    if i%20 == 0:
        print(f"Saved {output_file}")

print("Processing complete!")

# Region Barcodes

In [ ]:
barcodes = pd.read_csv(
    path_data + 'matrix_files/T1_E12_71/T1_E12_71_raw/barcodes.tsv.gz',
    sep='\t',
    header=None,
    index_col=0
)
barcodes.index.name = None
barcodes['x'] = pd.Series(index=barcodes.index.tolist())
barcodes['y'] = pd.Series(index=barcodes.index.tolist())

In [ ]:
barcodes_list = barcodes.index.tolist()

for inst_file in glob(path_landscape_files + 'map_parquet_files/*.parquet'):

    inst_chunk = pd.read_parquet(inst_file)

    common_barcodes = list(set(inst_chunk.index.tolist()).intersection(barcodes_list))

    print(inst_file, 'found', len(common_barcodes), 'barcodes')

    if len(common_barcodes) > 0:
        barcodes.loc[common_barcodes, 'x'] = inst_chunk.loc[common_barcodes, 1]
        barcodes.loc[common_barcodes, 'y'] = inst_chunk.loc[common_barcodes, 2]


In [ ]:
barcodes.head()

In [ ]:
barcodes.to_parquet(path_landscape_files + 'meta_spots.parquet')

## Cells

In [42]:
path_landscape_files

'data/landscape_files/IST_Substrate_E12_71_2025-07-31_v2'

In [36]:
cells = pd.read_csv(
    path_data + '/matrix_files/' + inst_slice + '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_cell_binned/barcodes.tsv.gz',
    sep='\t',
    header=None,
    index_col=0
)

In [37]:
cells.head()

""
0
cell100000:11721:24048
cell100001:11837:24263
cell100002:12260:23447
cell100003:11841:24234
cell100004:12301:23881


In [38]:
high_res_scale = 1/0.382

high_res_scale = 1

In [39]:
gc = pd.read_csv(path_data + 'registered_images/globalpos_' + dataset_name + '.csv', index_col=0)
gc

,X_shift,Y_shift
Sample_ID,,
T1_E12_71,4156,6187
T2_E12_71,3665,12827
T3_E12_71,2718,20134
T4_E12_71,1959,29518
T5_E12_71,2109,37553
T6_E12_71,9420,4390
T7_E12_71,9045,11593
T8_E12_71,8631,20424
T9_E12_71,8188,28995


In [40]:
tmp = pd.DataFrame([x.split(':') for x in cells.index.tolist()])
tmp.set_index(0, inplace=True)
tmp.index.name = None
tmp.columns = ['x', 'y']
tmp = tmp.astype(float)
tmp['x'] = (tmp['x'] - gc.loc[inst_slice + '_' + dataset_name, 'X_shift']) * high_res_scale
tmp['y'] = (tmp['y'] - gc.loc[inst_slice + '_' + dataset_name, 'Y_shift']) * high_res_scale

tmp["geometry"] = tmp.apply(
    # swapped for some reason
        lambda row: [row["x"], row["y"]] , axis=1
    )

tmp['name'] = pd.Series(tmp.index.tolist(), index=tmp.index.tolist())

tmp[['name', 'geometry']].to_parquet(path_landscape_files + 'cell_metadata.parquet')

tmp

,x,y,geometry,name
cell100000,3090.0,3624.0,"[3090.0, 3624.0]",cell100000
cell100001,3206.0,3839.0,"[3206.0, 3839.0]",cell100001
cell100002,3629.0,3023.0,"[3629.0, 3023.0]",cell100002
cell100003,3210.0,3810.0,"[3210.0, 3810.0]",cell100003
cell100004,3670.0,3457.0,"[3670.0, 3457.0]",cell100004
...,...,...,...,...
cell99999,3640.0,3002.0,"[3640.0, 3002.0]",cell99999
cell9999,1784.0,2643.0,"[1784.0, 2643.0]",cell9999
cell999,1573.0,1748.0,"[1573.0, 1748.0]",cell999
cell99,1240.0,1897.0,"[1240.0, 1897.0]",cell99


In [ ]:
print(tmp.x.min(), tmp.x.max())
print(tmp.y.min(), tmp.y.max())

In [ ]:
clusters = pd.DataFrame(index=tmp.index.tolist())
clusters['cluster'] = pd.Series(0, index=tmp.index.tolist())

In [ ]:
output_dir = Path(path_landscape_files + 'cell_clusters')
output_dir.mkdir(parents=True, exist_ok=True)
clusters.to_parquet(path_landscape_files + 'cell_clusters/cluster.parquet')

## Segmented Cells

In [ ]:
tile_bounds = {}
tile_bounds["x_min"] = 0
tile_bounds["x_max"] = 20000
tile_bounds["y_min"] = 0
tile_bounds["y_max"] = 20000

In [ ]:
poly = pd.read_csv(path_data + 'cell_masks/' + inst_slice + '_' + dataset_name + '_Expanded_5um_cell_contour_coords.csv')

poly['vertex_x'] = (poly['vertex_x'] - gc.loc[inst_slice + '_' + dataset_name, 'Y_shift']) * high_res_scale
poly['vertex_y'] = (poly['vertex_y'] - gc.loc[inst_slice + '_' + dataset_name, 'X_shift']) * high_res_scale

poly.head()

In [ ]:
# Group by 'cell_id' and aggregate the coordinates into lists
grouped = poly.groupby("cell_id").agg(list)

def safe_polygon(row):
    try:
        return Polygon(zip(row["vertex_x"], row["vertex_y"]))
    except Exception as e:
        # print(f"Error processing row {row.name}: {e}")
        return Polygon()

# # Create a new column for polygons
# grouped["geometry"] = grouped.apply(
#     lambda row: Polygon(zip(row["vertex_x"], row["vertex_y"])), axis=1
# )

grouped["geometry"] = grouped.apply(safe_polygon, axis=1)

# Convert the DataFrame with polygon data into a GeoDataFrame
cells = gpd.GeoDataFrame(grouped, geometry="geometry")[["geometry"]]


In [ ]:
def simple_format(geometry, image_scale):
    # factor in scaling
    return [[[coord[0] / image_scale, coord[1] / image_scale] for coord in polygon] for polygon in geometry]

In [ ]:
def transform_polygon(polygon):

    exterior_coords = polygon.exterior.coords

    # Creating the original structure by directly using numpy array for each coordinate pair
    original_format_coords = np.array([np.array(coord) for coord in exterior_coords])

    return np.array([original_format_coords], dtype=object)

In [ ]:
# Apply the transformation to each polygon
cells["NEW_GEOMETRY"] = cells["geometry"].apply(
    lambda poly: transform_polygon(poly)
)

In [ ]:
cells["GEOMETRY"] = cells["NEW_GEOMETRY"].apply(lambda x: simple_format(x, image_scale))

cells["polygon"] = cells["GEOMETRY"].apply(lambda x: Polygon(x[0]))

gdf_cells = gpd.GeoDataFrame(geometry=cells["polygon"])

gdf_cells["center_x"] = gdf_cells.centroid.x
gdf_cells["center_y"] = gdf_cells.centroid.y

In [ ]:
gdf_cells.head()

In [ ]:
output_dir = Path(path_landscape_files + 'cell_segmentation')
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
tile_size = 250
tile_size_x = tile_size
tile_size_y = tile_size

In [ ]:
# if not os.path.exists(path_output):
#     os.mkdir(path_output)

x_min = tile_bounds["x_min"]
x_max = tile_bounds["x_max"]
y_min = tile_bounds["y_min"]
y_max = tile_bounds["y_max"]

# Calculate the number of tiles needed
n_tiles_x = int(np.ceil((x_max - x_min) / tile_size_x))
n_tiles_y = int(np.ceil((y_max - y_min) / tile_size_y))
print(n_tiles_x, n_tiles_y)

In [ ]:
path_output = path_landscape_files + 'cell_segmentation'

In [ ]:
for i in range(n_tiles_x):

    if i % 2 == 0:
        print('row', i)

    for j in range(n_tiles_y):
        tile_x_min = x_min + i * tile_size_x
        tile_x_max = tile_x_min + tile_size_x
        tile_y_min = y_min + j * tile_size_y
        tile_y_max = tile_y_min + tile_size_y

        # find cell polygons with centroids in the tile
        keep_cells = gdf_cells[
            (gdf_cells.center_x >= tile_x_min)
            & (gdf_cells.center_x < tile_x_max)
            & (gdf_cells.center_y >= tile_y_min)
            & (gdf_cells.center_y < tile_y_max)
        ].index.tolist()

        inst_geo = cells.loc[keep_cells, ["GEOMETRY"]]

        # try adding cell name to geometry
        inst_geo["name"] = pd.Series(
            inst_geo.index.tolist(), index=inst_geo.index.tolist()
        )

        filename = f"{path_output}/cell_tile_{i}_{j}.parquet"

        # Save the filtered DataFrame to a Parquet file
        if inst_geo.shape[0] > 0:
            inst_geo[["GEOMETRY", "name"]].to_parquet(filename)

## Pseudo-Transcript Jitter

In [ ]:
path_data

In [ ]:
ls data/IST_data/

In [ ]:
dataset_name

In [ ]:
adata = sc.read_10x_mtx(path_data + 'matrix_files/' + inst_slice +  '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_raw/')
adata

In [ ]:
adata_cell = sc.read_10x_mtx(path_data + 'matrix_files/' + inst_slice +  '_' + dataset_name + '/' + inst_slice + '_' + dataset_name + '_cell_binned/')
adata_cell

## Meta Gene

In [ ]:
list_genes = adata.var.index.tolist()
meta_gene = pd.DataFrame(index=list_genes)
from matplotlib.colors import to_hex
# Get all categorical color palettes from Matplotlib and flatten them into a single list of colors
palettes = [plt.get_cmap(name).colors for name in plt.colormaps() if "tab" in name]
flat_colors = [color for palette in palettes for color in palette]

# Convert RGB tuples to hex codes
flat_colors_hex = [to_hex(color) for color in flat_colors]

# Use modular arithmetic to assign a color to each gene, white for genes with "Blank"
colors = [
    flat_colors_hex[i % len(flat_colors_hex)] if "Blank" not in gene else "#FFFFFF"
    for i, gene in enumerate(list_genes)
]

# Create a DataFrame with genes and their assigned colors
ser_color = pd.Series(colors, index=list_genes)

In [ ]:
meta_gene['mean'] = pd.Series(100, index=list_genes)
meta_gene['std'] = pd.Series(10, index=list_genes)
meta_gene['max'] = pd.Series(100, index=list_genes)
meta_gene['non-zero'] = pd.Series(0.5, index=list_genes)
meta_gene['color'] = ser_color

In [ ]:
path_landscape_files

In [ ]:
meta_gene.to_parquet(path_landscape_files + 'meta_gene.parquet')

## Landscape Visualization 

In [4]:
server_address = dega.viz.get_local_server()

In [5]:
server_address

53696

In [13]:
# landscape = dega.viz.Landscape(
#     technology='Xenium',
#     base_url = f'http://localhost:{server_address}/data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs'
# )
# landscape

In [15]:
ls data/landscape_files/IST_Substrate_E12_71_2025-07-29_v2/

cbg/
cell_boundaries.parquet
cell_clusters/
cell_metadata.parquet
cell_segmentation/
landscape_parameters.json
messed_up_polars_format_transcript_tiles/
meta_gene.parquet
micron_to_image_transform.csv
pyramid_images/
spot_positions.parquet
transcript_tiles/
v2_transcript_tiles/


In [20]:
landscape = dega.viz.Landscape(
    technology='Xenium',
    # base_url = f'http://localhost:{server_address}/data/IST_landscape_files'
    base_url = f'http://localhost:{server_address}/data/landscape_files/IST_Substrate_E12_71_2025-07-29_v2'
    # base_url = f'http://localhost:{server_address}/data/landscape_files/IST_Substrate_E12_71_2025-08-01-wh'
    # base_url = f'http://localhost:{server_address}/data/landscape_files/IST_tmp'
    
)
landscape

Landscape(base_url='http://localhost:53696/data/landscape_files/IST_tmp', cell_attr=['leiden'], technology='Xe…

In [ ]:
landscape_parameters = {
    "technology": "Xenium",
    "segmentation_approach": [
        "default"
    ],
    "max_pyramid_zoom": 16,
    "tile_size": 250,
    "image_info": [
        {
            "name": "h&e",
            "button_name": "H",
            "color": [
                0,
                0,
                255
            ]
        },
        {
            "name": "bound",
            "button_name": "BOUND",
            "color": [
                0,
                255,
                0
            ]
        },
        {
            "name": "rna",
            "button_name": "RNA",
            "color": [
                255,
                0,
                0
            ]
        },
        {
            "name": "prot",
            "button_name": "PROT",
            "color": [
                255,
                255,
                255
            ]
        }
    ],
    "image_format": ".webp",
    "use_int_index": true
}

In [ ]:
### Save Landscape Parameters

In [ ]:
from pathlib import Path
import json

path = Path("data/landscape_files/IST_mouse_cranium/landscape_parameters.json")

# Make sure directory exists (should be true, but this is safe)
path.parent.mkdir(parents=True, exist_ok=True)

# Write the JSON to file
with path.open("w") as f:
    json.dump(landscape_parameters, f, indent=2)


## Meta Cluster

In [ ]:
pd.read_parquet('data/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs/cell_clusters/meta_cluster.parquet').head()

In [ ]:
meta_cluster = pd.DataFrame()
meta_cluster.loc['0', 'color'] = '#ff7f0e'
meta_cluster.loc['0', 'count'] = 1000
meta_cluster.to_parquet(path_landscape_files + 'cell_clusters/meta_cluster.parquet')

In [ ]:
# pd.read_parquet(path_landscape_files + 'cell_clusters/cluster.parquet')